# Silver Layer — poe (provider order entry)

## READINESS: profiled, rules drafted, awaiting team approval

The profiling cells below were run read-only against Bronze on 2026-10-20 (results in
**Profiling Findings**) and the Silver rules derived from them are implemented. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written.

**Known publish blocker.** The current `cdr_mimiciv.silver.poe` (written 2026-10-13, every
column STRING) cannot be read: every query fails with `UC_DEPENDENCY_DOES_NOT_EXIST`
("one of the underlying resources does not exist"). `datetimeevents` had the same error, and
`createOrReplace` failed on it. The publish cell checks the target first and stops with a
clear message if it is broken. This notebook never drops a table: removing the broken
registration is a separate, explicitly approved step.

## Source
`<source_schema>.poe` (default `cdr_mimiciv.bronze.poe`)

## Target
`<target_schema>.poe` (default `cdr_mimiciv.silver.poe`), published with CREATE OR REPLACE
semantics. No tables are dropped.

## Grain (official MIMIC-IV hosp documentation, `poe`)
Provider order entry is the interface through which care providers enter orders; most
treatments and procedures must be ordered via POE. One row per order, identified by `poe_id`,
which is composed of `subject_id` and `poe_seq` as `subject_id-poe_seq`. `poe_seq` is a
monotonically increasing integer documented to sort a patient's orders chronologically.
Documented fields: `poe_id`, `poe_seq`, `subject_id`, `hadm_id`, `ordertime` (when the order
was made), `order_type` (16 documented values), `order_subtype` (best read with
`order_type`), `transaction_type` (documented: Change, Co, D/C, H, New, T),
`discontinue_of_poe_id` (the earlier order this order discontinues), `discontinued_by_poe_id`
(the later order that discontinued this one), `order_provider_id` (anonymous provider id) and
`order_status` ('Active' or 'Inactive').
Documented NOT NULL: `poe_id`, `poe_seq`, `subject_id`, `ordertime`, `order_type`.
Links to `poe_detail` on `poe_id`.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions and labevents: preserve the source value, identify the anomaly,
add a DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order. Bronze is already typed
  (INT ids, TIMESTAMP `ordertime`), so the old re-cast to BIGINT is removed.
- Add DQ flags (our engineering choice): `dq_hadm_not_in_admissions`,
  `dq_discontinue_of_poe_id_not_in_poe`, `dq_discontinued_by_poe_id_not_in_poe`,
  `dq_ordertime_before_previous_seq`.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no row removal.

## Cost note
`poe` has about 52M rows. Profiling uses one aggregate pass plus a few anti-joins on
distinct keys. The `poe_seq` chronology flag needs one window over `subject_id`, which is the
only shuffle in the build.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "false")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F
from pyspark.sql import Window

SOURCE_TABLE = f"{BRONZE}.poe"
TARGET_TABLE = f"{SILVER}.poe"
KEY = "poe_id"

DOCUMENTED_COLUMNS = [
    "poe_id", "poe_seq", "subject_id", "hadm_id", "ordertime", "order_type",
    "order_subtype", "transaction_type", "discontinue_of_poe_id",
    "discontinued_by_poe_id", "order_provider_id", "order_status"
]
DOCUMENTED_ORDER_TYPES = [
    "ADT orders", "Blood Bank", "Cardiology", "Consults", "Critical Care", "General Care",
    "Hemodialysis", "IV therapy", "Lab", "Medications", "Neurology", "Nutrition", "OB",
    "Radiology", "Respiratory", "TPN"
]
DOCUMENTED_TRANSACTION_TYPES = ["Change", "Co", "D/C", "H", "New", "T"]
DOCUMENTED_ORDER_STATUS = ["Active", "Inactive"]

In [0]:
poe_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = poe_bronze.columns

poe_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
display(poe_bronze.limit(10))

root
 |-- poe_id: string (nullable = true)
 |-- poe_seq: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- ordertime: timestamp (nullable = true)
 |-- order_type: string (nullable = true)
 |-- order_subtype: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- discontinue_of_poe_id: string (nullable = true)
 |-- discontinued_by_poe_id: string (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- order_status: string (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []


poe_id,poe_seq,subject_id,hadm_id,ordertime,order_type,order_subtype,transaction_type,discontinue_of_poe_id,discontinued_by_poe_id,order_provider_id,order_status
17113640-71,71,17113640,21514262,2174-09-18T15:10:10.000Z,ADT orders,Discharge,New,null,null,P80BM1,Inactive
17113640-72,72,17113640,23896135,2175-03-11T03:04:46.000Z,Lab,null,New,null,null,P13CBL,Inactive
17113640-73,73,17113640,23896135,2175-03-11T03:05:07.000Z,Lab,null,New,null,null,P13CBL,Inactive
17113640-74,74,17113640,23896135,2175-03-11T03:32:58.000Z,Lab,null,New,null,null,P13CBL,Inactive
17113640-75,75,17113640,23896135,2175-03-11T03:39:50.000Z,Lab,null,New,null,null,P13CBL,Inactive
17113640-80,80,17113640,23896135,2175-03-11T06:31:12.000Z,Medications,null,New,null,null,P363OG,Inactive
17113640-81,81,17113640,23896135,2175-03-11T06:31:12.000Z,Medications,null,New,null,null,P363OG,Inactive
17113640-82,82,17113640,23896135,2175-03-11T06:31:12.000Z,Medications,null,New,null,null,P363OG,Inactive
17113640-83,83,17113640,23896135,2175-03-11T07:16:02.000Z,Medications,null,New,null,null,P13CBL,Inactive
17113640-84,84,17113640,23896135,2175-03-11T07:16:02.000Z,Medications,null,New,null,null,P13CBL,Inactive


In [0]:
%sql
Select distinct order_provider_id
from cdr_mimiciv.bronze.poe;

order_provider_id
P85CN8
P04G0V
P12XYO
P55WEI
P5719V
P35094
P849KX
P222AW
P95DQS
P26VDQ


In [0]:
# single-pass profile: grain/key, missingness of every column, poe_id composition,
# documented value domains, ordertime range (no repeated scans, no shuffles)
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

profile_exprs = [
    F.count("*").alias("total_rows"),
    F.countDistinct(KEY).alias("distinct_poe_id"),
]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.col("poe_id") != F.concat_ws("-", F.col("subject_id").cast("string"), F.col("poe_seq").cast("string")))
        .alias("poe_id_not_subject_dash_seq"),
    _cnt(~F.col("order_type").isin(DOCUMENTED_ORDER_TYPES)).alias("order_type_undocumented"),
    _cnt(~F.col("transaction_type").isin(DOCUMENTED_TRANSACTION_TYPES)).alias("transaction_type_undocumented"),
    _cnt(~F.col("order_status").isin(DOCUMENTED_ORDER_STATUS)).alias("order_status_undocumented"),
    _cnt(F.trim(F.col("order_subtype")) == "").alias("order_subtype_empty_string"),
    _cnt((F.col("transaction_type") == "New") & F.col("discontinue_of_poe_id").isNotNull())
        .alias("new_order_with_discontinue_of"),
    F.min("ordertime").alias("min_ordertime"),
    F.max("ordertime").alias("max_ordertime"),
]

poe_profile = poe_bronze.agg(*profile_exprs).first().asDict()
for k, v in poe_profile.items():
    print(f"{k}: {v}")

total_rows: 52212109
distinct_poe_id: 52212109
null_poe_id: 0
null_poe_seq: 0
null_subject_id: 0
null_hadm_id: 0
null_ordertime: 0
null_order_type: 0
null_order_subtype: 32220166
null_transaction_type: 0
null_discontinue_of_poe_id: 41480217
null_discontinued_by_poe_id: 41556491
null_order_provider_id: 308039
null_order_status: 0
poe_id_not_subject_dash_seq: 0
order_type_undocumented: 0
transaction_type_undocumented: 0
order_status_undocumented: 0
order_subtype_empty_string: 0
new_order_with_discontinue_of: 3
min_ordertime: 2105-10-04 17:34:15
max_ordertime: 2214-12-24 13:29:21


In [0]:
# duplicates: derived from the aggregate; drill down only if the key is not unique
non_null_ids = poe_profile["total_rows"] - poe_profile[f"null_{KEY}"]
print("poe_id duplicates present:", poe_profile["distinct_poe_id"] != non_null_ids)

if poe_profile["distinct_poe_id"] != non_null_ids:
    duplicate_ids = poe_bronze.groupBy(KEY).count().filter(F.col("count") > 1)
    print("Duplicate poe_id values:", duplicate_ids.count())
    display(poe_bronze.join(duplicate_ids.limit(50).select(KEY), KEY, "inner").orderBy(KEY))

poe_id duplicates present: False


In [0]:
# categorical values (small result sets)
for c in ["order_type", "transaction_type", "order_status"]:
    print(f"=== {c} ===")
    display(poe_bronze.groupBy(c).count().orderBy(F.desc("count")))

print("=== order_type x order_subtype ===")
display(
    poe_bronze.groupBy("order_type", "order_subtype").count()
    .orderBy("order_type", F.desc("count"))
)

=== order_type ===


order_type,count
Medications,23295778
Lab,8908993
General Care,7670925
ADT orders,2776808
IV therapy,2747821
Nutrition,2131521
Radiology,1876077
Consults,826947
Blood Bank,666209
Respiratory,608476


=== transaction_type ===


transaction_type,count
New,41468286
Change,5714162
D/C,5017727
H,6903
T,5031


=== order_status ===


order_status,count
Inactive,52059806
Active,152303


=== order_type x order_subtype ===


order_type,order_subtype,count
ADT orders,Discharge,1390522
ADT orders,Admit,1019546
ADT orders,Transfer,214726
ADT orders,Postop,152014
Blood Bank,Blood tests,409582
Blood Bank,Red Cell Product Order,166592
Blood Bank,Platelet Product Order,51099
Blood Bank,Frozen Plasma Product Order,31350
Blood Bank,Cryoprecipitate Product Order,5146
Blood Bank,Derivative Order,2440


In [0]:
# chronology: documented that poe_seq sorts a patient's orders chronologically
w_seq = Window.partitionBy("subject_id").orderBy("poe_seq")
print(
    "orders whose ordertime is earlier than the previous poe_seq's ordertime:",
    poe_bronze.select(
        "ordertime", F.lag("ordertime").over(w_seq).alias("_prev_ordertime")
    ).where(F.col("ordertime") < F.col("_prev_ordertime")).count()
)

orders whose ordertime is earlier than the previous poe_seq's ordertime: 276992


In [0]:
# relationships (observational): discontinuation links, patients, admissions, providers
order_ids = poe_bronze.select(F.col(KEY).alias("_ref_poe_id"))

for link in ["discontinue_of_poe_id", "discontinued_by_poe_id"]:
    print(
        f"rows whose {link} is not a poe_id in poe:",
        poe_bronze.where(F.col(link).isNotNull())
        .join(order_ids, F.col(link) == F.col("_ref_poe_id"), "left_anti").count()
    )

discontinued = poe_bronze.select(
    F.col(KEY).alias("_d_poe_id"), F.col("ordertime").alias("_d_ordertime")
)
print(
    "orders that discontinue an order placed after them:",
    poe_bronze.join(discontinued, F.col("discontinue_of_poe_id") == F.col("_d_poe_id"), "inner")
    .where(F.col("ordertime") < F.col("_d_ordertime")).count()
)

patients_table = f"{SILVER}.patients"
admissions_table = f"{SILVER}.admissions"
provider_table = f"{SILVER}.provider"

if spark.catalog.tableExists(patients_table):
    print(
        "distinct poe subject_id not found in patients:",
        poe_bronze.select("subject_id").distinct()
        .join(spark.table(patients_table).select("subject_id").distinct(), "subject_id", "left_anti").count()
    )

if spark.catalog.tableExists(admissions_table):
    admissions = spark.table(admissions_table)
    orphan_pairs = (
        poe_bronze.where(F.col("hadm_id").isNotNull())
        .groupBy("subject_id", "hadm_id").count()
        .join(admissions.select("subject_id", "hadm_id").distinct(), ["subject_id", "hadm_id"], "left_anti")
    )
    print("(subject_id, hadm_id) pairs not found in admissions:")
    display(orphan_pairs)

    # ordertime relative to the admission (observational; orders before admission are expected
    # for ED and pre-admission orders, so no flag)
    display(
        poe_bronze.join(admissions.select("subject_id", "hadm_id", "admittime", "dischtime", "edregtime"),
                        ["subject_id", "hadm_id"], "inner")
        .agg(
            _cnt(F.col("ordertime") < F.col("admittime")).alias("before_admittime"),
            _cnt(F.col("ordertime") < F.coalesce("edregtime", "admittime")).alias("before_ed_or_admittime"),
            _cnt(F.col("ordertime") > F.col("dischtime")).alias("after_dischtime"),
            _cnt(F.col("ordertime") > F.col("dischtime") + F.expr("INTERVAL 1 DAY")).alias("after_dischtime_1d"),
        )
    )

if spark.catalog.tableExists(provider_table):
    print(
        "distinct order_provider_id not found in provider:",
        poe_bronze.where(F.col("order_provider_id").isNotNull()).select("order_provider_id").distinct()
        .join(spark.table(provider_table).select(F.col("provider_id").alias("order_provider_id")).distinct(),
              "order_provider_id", "left_anti").count()
    )

rows whose discontinue_of_poe_id is not a poe_id in poe: 38501
rows whose discontinued_by_poe_id is not a poe_id in poe: 1667
orders that discontinue an order placed after them: 65
distinct poe subject_id not found in patients: 0
(subject_id, hadm_id) pairs not found in admissions:


subject_id,hadm_id,count
17080681,25029676,95


before_admittime,before_ed_or_admittime,after_dischtime,after_dischtime_1d
4634129,955889,78401,604


distinct order_provider_id not found in provider: 0


## Profiling Findings

Recorded from read-only SQL against `cdr_mimiciv.bronze.poe` on 2026-10-20 (Bronze last
updated 2026-10-20 01:32 UTC). The cells above reproduce every check.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows / distinct `poe_id` | 52,212,109 / 52,212,109 | Key unique; no deduplication |
| `poe_id` = `subject_id-poe_seq` [DOC] | 0 mismatches | None |
| NULL `poe_id`, `poe_seq`, `subject_id`, `ordertime`, `order_type`, `transaction_type`, `order_status` | 0 | None |
| NULL `hadm_id` | 0 | None (documented nullable) |
| NULL `order_subtype` | 32,220,166 (61.7%); mostly Medications and Lab, which have almost no subtypes | Keep |
| NULL `order_provider_id` | 308,039 (0.6%) | Keep |
| NULL `discontinue_of_poe_id` / `discontinued_by_poe_id` | 41,480,217 / 41,556,491 | Keep; NULL means no link [DOC] |
| `ordertime` range | 2105-10-04 to 2214-12-24 | Shifted dates [DOC]; no correction |
| `order_type` | 16 values, all documented | None |
| `transaction_type` | New 41,468,286; Change 5,714,162; D/C 5,017,727; H 6,903; T 5,031; `Co` not present | None |
| `order_status` | Inactive 52,059,806; Active 152,303 | None |
| `poe_seq` order vs `ordertime` | 276,992 orders charted earlier than the patient's previous `poe_seq` | `dq_ordertime_before_previous_seq` [OURS]; contradicts the documented chronological sort [DOC]; no reordering |
| `discontinue_of_poe_id` not in poe | 38,501 rows | `dq_discontinue_of_poe_id_not_in_poe` [OURS]; link kept |
| `discontinued_by_poe_id` not in poe | 1,667 rows | `dq_discontinued_by_poe_id_not_in_poe` [OURS]; link kept |
| Order discontinues an order placed after it | 65 | Recorded only (too few to flag) |
| `transaction_type` New with `discontinue_of_poe_id` | 3 | Recorded only |
| `subject_id` not in patients | 0 distinct | None |
| `(subject_id, hadm_id)` not in admissions | 1 pair (17080681, 25029676), 95 rows; the same pair as in labevents | `dq_hadm_not_in_admissions` [OURS]; rows kept |
| `order_provider_id` not in provider | 0 distinct | None |
| `ordertime` before `admittime` / before ED registration or admittime | 4,634,129 / 955,889 | Recorded only: pre-admission and ED orders are expected; not flagged |
| `ordertime` after `dischtime` / more than 1 day after | 78,401 / 604 | Recorded only |
| Exact duplicate rows | 0 (implied: `poe_id` unique) | None |

In [0]:
# Silver transformation: all source columns unchanged + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed

# distinct admission keys from Silver admissions (small; drives dq_hadm_not_in_admissions)
admissions_keys = (
    spark.table(f"{SILVER}.admissions")
    .select("subject_id", "hadm_id").distinct()
    .withColumn("_adm_match", F.lit(True))
)

# all order ids, used to check both discontinuation links
poe_ids_for_of = poe_bronze.select(F.col(KEY).alias("_of_poe_id"), F.lit(True).alias("_of_match"))
poe_ids_for_by = poe_bronze.select(F.col(KEY).alias("_by_poe_id"), F.lit(True).alias("_by_match"))

w_seq = Window.partitionBy("subject_id").orderBy("poe_seq")

poe_silver = (
    poe_bronze.select(*SOURCE_COLUMNS)
    .join(F.broadcast(admissions_keys), ["subject_id", "hadm_id"], "left")
    .join(poe_ids_for_of, F.col("discontinue_of_poe_id") == F.col("_of_poe_id"), "left")
    .join(poe_ids_for_by, F.col("discontinued_by_poe_id") == F.col("_by_poe_id"), "left")
    .withColumn("_prev_ordertime", F.lag("ordertime").over(w_seq))
    .select(
        *SOURCE_COLUMNS,
        (F.col("hadm_id").isNotNull() & F.col("_adm_match").isNull())
            .alias("dq_hadm_not_in_admissions"),
        (F.col("discontinue_of_poe_id").isNotNull() & F.col("_of_match").isNull())
            .alias("dq_discontinue_of_poe_id_not_in_poe"),
        (F.col("discontinued_by_poe_id").isNotNull() & F.col("_by_match").isNull())
            .alias("dq_discontinued_by_poe_id_not_in_poe"),
        F.coalesce(F.col("ordertime") < F.col("_prev_ordertime"), F.lit(False))
            .alias("dq_ordertime_before_previous_seq"),
        F.current_timestamp().alias("_silver_processed_at"),
    )
)

expected_schema = {f.name: f.dataType.simpleString() for f in poe_bronze.schema.fields}
expected_schema.update({
    "dq_hadm_not_in_admissions": "boolean",
    "dq_discontinue_of_poe_id_not_in_poe": "boolean",
    "dq_discontinued_by_poe_id_not_in_poe": "boolean",
    "dq_ordertime_before_previous_seq": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (read-only SQL, 2026-10-20)
PROFILED_COUNTS = {
    "silver_rows": 52212109,
    "dq_hadm_not_in_admissions": 95,
    "dq_discontinue_of_poe_id_not_in_poe": 38501,
    "dq_discontinued_by_poe_id_not_in_poe": 1667,
    "dq_ordertime_before_previous_seq": 276992,
}

poe_silver.printSchema()

root
 |-- poe_id: string (nullable = true)
 |-- poe_seq: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- ordertime: timestamp (nullable = true)
 |-- order_type: string (nullable = true)
 |-- order_subtype: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- discontinue_of_poe_id: string (nullable = true)
 |-- discontinued_by_poe_id: string (nullable = true)
 |-- order_provider_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_hadm_not_in_admissions: boolean (nullable = false)
 |-- dq_discontinue_of_poe_id_not_in_poe: boolean (nullable = false)
 |-- dq_discontinued_by_poe_id_not_in_poe: boolean (nullable = false)
 |-- dq_ordertime_before_previous_seq: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "poe is not approved for publication: run and review the profiling cells, "
        "then set widget profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in poe_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"
assert KEY in pre_schema, f"Key column {KEY} missing"

# the target must be readable (or absent) before CREATE OR REPLACE; a broken registration
# (UC_DEPENDENCY_DOES_NOT_EXIST) makes createOrReplace fail, and this notebook never drops tables
if spark.catalog.tableExists(TARGET_TABLE):
    try:
        spark.table(TARGET_TABLE).limit(1).collect()
    except Exception as e:
        raise RuntimeError(
            f"{TARGET_TABLE} exists but cannot be read, so it cannot be replaced: {str(e)[:300]}. "
            "Removing the broken table is a separate step that needs explicit approval."
        )

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    poe_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.poe


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct(KEY).alias("distinct_key"),
    F.sum(F.when(F.col(KEY).isNull(), 1).otherwise(0)).alias("null_key"),
    F.sum(F.when(F.col("_silver_processed_at").isNull(), 1).otherwise(0)).alias("null_processed_at"),
    F.sum(F.when(F.col("dq_hadm_not_in_admissions"), 1).otherwise(0)).alias("dq_hadm_not_in_admissions"),
    F.sum(F.when(F.col("dq_discontinue_of_poe_id_not_in_poe"), 1).otherwise(0)).alias("dq_discontinue_of_poe_id_not_in_poe"),
    F.sum(F.when(F.col("dq_discontinued_by_poe_id_not_in_poe"), 1).otherwise(0)).alias("dq_discontinued_by_poe_id_not_in_poe"),
    F.sum(F.when(F.col("dq_ordertime_before_previous_seq"), 1).otherwise(0)).alias("dq_ordertime_before_previous_seq"),
).first().asDict()
bronze_rows = poe_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_key"] == 0, f"{KEY} contains NULLs"
assert post["distinct_key"] == post["silver_rows"], f"{KEY} is not unique"
assert post["null_processed_at"] == 0
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("poe:OK")